In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
df=pd.read_csv("messy_movies_5000_plus.csv")
df=df.sort_values(by='Movie_ID',ascending=True)
df.info()
df.isna().mean()

In [ ]:
# Removing White Spaces and Inconsistency
df['Title']=df['Title'].str.strip().str.title()
df['Genre']=df['Genre'].str.strip().replace({'comedy':'Comedy','Sci-Fi':'SciFi','action':'Action','HORROR':'Horror','ACTION':'Action'})
df['Language'] = df['Language'].str.strip().str.title()
df['Country']=df['Country'].str.strip().str.title()
df['Director']=df['Director'].str.strip().str.title()


In [ ]:

#Checking Is there any Values which Lies outside the 0 to 10 Range in IMDB AND 0 to 100 in Rotten Tomatoes
# Invalid= df[~df['IMDB_Rating'].between(0,10)] 
# df.loc[~df['RottenTomatoes'].between(0,100), 'RottenTomatoes'].unique()
df['IMDB_Rating'] = pd.to_numeric(df['IMDB_Rating'], errors='coerce')
df['RottenTomatoes'] = pd.to_numeric(df['RottenTomatoes'].str.replace('%',''),errors='coerce')

df = df.drop_duplicates(subset='Movie_ID')


In [ ]:
# Cleaning Box_Office and Budget by removing "M",",","$"
for col in ['Box_Office','Budget']:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace('$','', regex=False)
        .str.replace(',','', regex=False)
        .str.replace('[Mm]','e6', regex=True)
    )
    df[col] = pd.to_numeric(df[col], errors='coerce')


In [ ]:
df['Release_Date'] = pd.to_datetime(df['Release_Date'].str.strip(),format='mixed',dayfirst=True,errors='coerce'
)
df.head(20)
df['Release_Date'].dtypes

In [ ]:
# Filling NA Values 
for col in ('IMDB_Rating','RottenTomatoes'):
    df[col]=df[col].fillna(df.groupby('Genre')[col].transform('mean'))

df['IMDB_Rating']=df['IMDB_Rating'].round(1)
df['RottenTomatoes']=df['RottenTomatoes'].round(0).astype(int)

df['Duration_Minutes']=df['Duration_Minutes'].fillna(df.groupby('Genre')['Duration_Minutes'].transform('median')).round(1)
df

In [ ]:
#df.isna().mean()*100

In [ ]:
# Some Meaningful Insights
Average_Rating_ByGenre=df.groupby('Genre')['IMDB_Rating'].mean().round(1)
#Average_Rating_ByGenre

Average_Rating_ByDirector=df.groupby('Director')['IMDB_Rating'].mean().round(1)
#Average_Rating_ByDirector

df['Release_Year'] = df['Release_Date'].dt.year
Average_Rating_ByYear=df.groupby('Release_Year')['IMDB_Rating'].mean().round(1)
#Average_Rating_ByYear

df['ROI'] = ((df['Box_Office'] - df['Budget']) / df['Budget']) * 100
df['Profit']=df['Box_Office']-df['Budget']
def Profit_Status(ROI):
    if ROI<=0:
        return 'Flop'
    elif ROI<50:
        return 'Average'
    elif ROI<=100:
        return 'Hit'
    elif ROI >100:
        return 'BlockBuster'
df['Profit_Status']=df['ROI'].apply(Profit_Status)


In [ ]:
# Bar Plot Showing average rating by Genre.
plt.figure(figsize=(8,6),facecolor='darkgrey')
sns.barplot(x=Average_Rating_ByGenre.index,y=Average_Rating_ByGenre.values,color='grey')
plt.xlabel('Genre Name')
plt.ylabel('Rating')
plt.title('Genres Rating Trend')
plt.grid(axis='y',linewidth=0.2,alpha=0.8,color='black')
plt.show()


In [ ]:

# Bar Plot Showing average rating by Director.
plt.figure(figsize=(8,6),facecolor='darkgrey')
sns.barplot(x=Average_Rating_ByDirector.index,y=Average_Rating_ByDirector.values,color='grey')
plt.xlabel('Director')
plt.ylabel('Ratings')
plt.title('Average Rating Of Directors')
plt.grid(axis='y',linewidth=0.4,alpha=0.4,color='black')
plt.tight_layout()
plt.show()

In [ ]:
# Line Plot Showing average rating Over Years.
plt.figure(figsize=(8,6),facecolor='darkgrey')
sns.lineplot(x=Average_Rating_ByYear.index,y=Average_Rating_ByYear.values,color='black',marker='o')
plt.xlabel('Years',color='black')
plt.ylabel('Ratings',color='black')
plt.xticks(color='black')
plt.yticks(color='black')
plt.title('Average Rating Per Year',color='black')
plt.grid(axis='y',linewidth=0.4,color='black')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,6),facecolor='darkgrey')
sns.scatterplot(data=df,x='Budget',y='IMDB_Rating',alpha=0.6,color='black')
plt.xlabel("Budget ($)")
plt.ylabel("IMDB Rating")
plt.title("Budget vs IMDB Rating")
plt.grid(True, linewidth=0.3, alpha=0.5)
plt.show()

In [ ]:
#Horizontal Bar Plot: Top-rated movies by rating
plt.figure(figsize=(8,6),facecolor='darkgrey')
Top10=df.sort_values('IMDB_Rating',ascending=False).head(10)
plt.barh(Top10['Title'],Top10['IMDB_Rating'],color='grey')
plt.xlabel("IMDB Rating",color="black")
plt.ylabel("Movie Title",color="black")
plt.xticks(color='black')
plt.yticks(color='black')
plt.gca().invert_yaxis()     # highest rating on top
plt.grid(axis='y', linewidth=0.5, alpha=0.5)
plt.title("Top10-Top Rated Movies",color="black")


In [ ]:
# Bar Plot Showing average rating by Genre.
Average_Profit_ByGenre=df.groupby('Genre')['Profit'].mean()
plt.figure(figsize=(8,6),facecolor='darkgrey')
sns.barplot(x=Average_Profit_ByGenre.index,y=Average_Profit_ByGenre.values,color='grey')
plt.xlabel('Genre Name')
plt.ylabel('Profit')
plt.title('Genres Profit Trend')
plt.grid(axis='y',linewidth=0.2,alpha=0.8,color='black')
plt.show()


In [ ]:
df.info()


In [ ]:
df.to_csv("cleaned_data_movies.csv", index=False)